# Experiment A: sample at T_melt/1.141 and compare with TURN

This notebook runs `src/experiment_a.py` from the Token Thermodynamics repository. The protocol, the decision rule and the budget are in `EXPERIMENT_A.md`. Read that first.

Settings to check before running. Turn on a GPU accelerator (T4 x2) and turn on internet access. For the gated Llama models, accept the licence on each model's Hugging Face page and add a Kaggle secret named `HF_TOKEN`.

In [ ]:
MODEL = "meta-llama/Llama-3.2-1B-Instruct"   # then meta-llama/Llama-3.2-3B-Instruct
TASK = "math"                                # "mbpp" executes model-written code
K = 32                                       # samples per question per temperature
SMOKE = True   # True: 8 problems, 4 samples, about 15 minutes, to check everything works.
               # Then set False and use Save Version, Save & Run All, for the real run.

if SMOKE:
    OUT = "/kaggle/working/results_smoke"    # kept apart so it never mixes with the real run
    EXTRA = "--n-problems 8 --k 4 --turn-samples 8 --max-new-tokens 256"
else:
    OUT = "/kaggle/working/results_a"
    EXTRA = ""

In [ ]:
!git clone -q https://github.com/pragyaangaur/Token-Thermodynamics.git
# TURN supplies the MATH split, the four-shot prompt, the answer parser and the grader.
# Pinned to the commit the harness was written against.
!git clone -q https://github.com/StigLidu/TURN.git && git -C TURN checkout -q 64b42e22762a05f4a5ef999c868c61a83c7adb2f
!pip install -q vllm sympy pylatexenc

In [ ]:
import os, glob, shutil
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret, gated models will fail:", e)

# Resuming. If an earlier version of this notebook's output is attached as an input,
# copy its checkpoints so finished phases and temperatures are skipped.
for prev in glob.glob("/kaggle/input/*/results_a"):
    shutil.copytree(prev, "/kaggle/working/results_a", dirs_exist_ok=True)
    print("resumed from", prev)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import vllm; print("vllm", vllm.__version__)

In [ ]:
%cd /kaggle/working/Token-Thermodynamics
# Two separate processes. The float32 pass frees its GPU memory when its process ends,
# so vLLM starts with the whole GPU. The second command reuses the saved melt phase.
!python src/experiment_a.py --model {MODEL} --task {TASK} --k {K} --turn-dir ../TURN \
    --out {OUT} --phases melt {EXTRA}
!python src/experiment_a.py --model {MODEL} --task {TASK} --k {K} --turn-dir ../TURN \
    --out {OUT} --dtype-gen half --tp 1 {EXTRA}

If vLLM fails to start on the T4, the likely cause is that its newest release no longer supports that GPU. Try `!pip install -q "vllm<0.10"`, restart the kernel and rerun. The melt phase is already saved and will not be repeated. The summary records which kind of log probabilities vLLM returned, so the entropy curve stays comparable to TURN's.

In [ ]:
!cat {OUT}/*/*/summary.json
!cd /kaggle/working && zip -qr results.zip $(basename {OUT}) && ls -la results.zip